# Trainability and classical baselines
**Quantum Machine Learning · Week 4 (QML for research) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/qml/w4d2.html

In [ ]:
# On Google Colab, this installs qiskit and qiskit-aer (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 qiskit-aer==0.17.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2 qiskit-aer==0.17.2

**Trainability and classical baselines.** The cures for barren plateaus (QC week 7) keep circuits small, shallow or
local, and circuits like that can often be imitated classically. Here is one imitation, built from Thursday's
Fourier picture: a classical surrogate that uses exactly the quantum model's frequencies.

In [ ]:
import time
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from sklearn.datasets import make_moons
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

estimator = AerEstimator(options={"backend_options": {"max_parallel_threads": 1}})
L = 3
x, w = ParameterVector("x", 2), ParameterVector("w", 4 * L)
circuit = QuantumCircuit(2)
for l in range(L):                                   # week 3's re-uploading classifier
    circuit.ry(x[0], 0)
    circuit.ry(x[1], 1)
    circuit.ry(w[4 * l], 0)
    circuit.rz(w[4 * l + 1], 0)
    circuit.ry(w[4 * l + 2], 1)
    circuit.rz(w[4 * l + 3], 1)
    circuit.cx(0, 1)

def scores(X, weight_sets):
    rows = np.array([np.concatenate([W, p]) for W in weight_sets for p in X])
    return estimator.run([(circuit, SparsePauliOp("ZI"), rows)]).result()[0].data.evs.reshape(len(weight_sets), len(X))

X, y = make_moons(n_samples=200, noise=0.15, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scaler = MinMaxScaler((-np.pi / 2, np.pi / 2)).fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

sigmoid = lambda z: 1 / (1 + np.exp(-z))
t0 = time.time()
theta = np.r_[0.6 * np.random.default_rng(0).normal(size=4 * L), 2.0, 0.0]
m, v = np.zeros_like(theta), np.zeros_like(theta)
shifts = [s * np.eye(4 * L)[j] for j in range(4 * L) for s in (np.pi / 2, -np.pi / 2)]
for k in range(1, 61):                               # week 3's training: cross-entropy, scale a and b, Adam
    W, a, b = theta[:-2], theta[-2], theta[-1]
    S = scores(X_train, [W] + [W + d for d in shifts])
    f, df = S[0], (S[1::2] - S[2::2]) / 2
    dz = (sigmoid(-a * f + b) - y_train) / len(y_train)
    g = np.r_[df @ (-a * dz), np.sum(-f * dz), np.sum(dz)]
    m, v = 0.9 * m + 0.1 * g, 0.999 * v + 0.001 * g ** 2
    theta = theta - 0.1 * (m / (1 - 0.9 ** k)) / (np.sqrt(v / (1 - 0.999 ** k)) + 1e-8)
W_q = theta[:-2]
predict_q = lambda X_: (sigmoid(-theta[-2] * scores(X_, [W_q])[0] + theta[-1]) > 0.5).astype(int)
print(f"quantum classifier: test accuracy {np.mean(predict_q(X_test) == y_test):.3f}, "
      f"{time.time() - t0:.1f} s to train, {60 * len(X_train) * (1 + 2 * 4 * L):,} circuit runs")

## The quantum model is a linear model in disguise
With 3 uploads per feature, the score is a Fourier series in both features with frequencies up to 3:
a combination of $\cos(\omega_1 x_1 + \omega_2 x_2)$ and $\sin(\omega_1 x_1 + \omega_2 x_2)$ for $\omega_1, \omega_2 \in \{-3, \dots, 3\}$.
That's 49 real features. Fit the trained circuit's scores with them by least squares.

In [ ]:
def fourier_features(X, L):
    pairs = [(a, b) for a in range(-L, L + 1) for b in range(-L, L + 1) if (a, b) > (0, 0)]   # one of each ± pair
    cols = [np.ones(len(X))]
    for a, b in pairs:
        phase = a * X[:, 0] + b * X[:, 1]
        cols += [np.cos(phase), np.sin(phase)]
    return np.stack(cols, axis=1)

probe = np.random.default_rng(1).uniform(-np.pi, np.pi, size=(300, 2))
F = fourier_features(probe, L)
z = scores(probe, [W_q])[0]
coef, *_ = np.linalg.lstsq(F, z, rcond=None)
print(f"{F.shape[1]} features; largest error of the least-squares fit: {np.abs(F @ coef - z).max():.1e}")
print("the five biggest coefficients:", np.round(np.sort(np.abs(coef))[::-1][:5], 3))

## The classical surrogate
Logistic regression on the same 49 features, with $C$ tuned by cross-validation on the training part.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

t0 = time.time()
search = GridSearchCV(LogisticRegression(max_iter=5000), {"C": [0.1, 1, 10, 100]}, cv=5)
search.fit(fourier_features(X_train, L), y_train)
acc_s = search.score(fourier_features(X_test, L), y_test)
print(f"classical surrogate: test accuracy {acc_s:.3f}, {time.time() - t0:.2f} s to tune and train, C = {search.best_params_['C']}")

## When surrogates get expensive
The menu grows with the number of features $d$: $(2L + 1)^d$ frequency vectors. Count them.

In [ ]:
for d in (2, 4, 8, 16):
    print(f"d = {d:2d} features, L = 3: {7 ** d:,} frequency vectors")

## Your turn: random Fourier features
When the full menu is too big, sample from it. Keep only $K$ random frequency pairs (plus the constant) and train
the surrogate on those. How many does it need to match the quantum classifier?

In [ ]:
pairs = [(a, b) for a in range(-L, L + 1) for b in range(-L, L + 1) if (a, b) > (0, 0)]
rng = np.random.default_rng(0)
# TODO: for K in (2, 4, 8, 16, 24): pick K pairs with rng.choice, build their cos and sin features,
#       fit the same cross-validated logistic regression as above, and print the test accuracy

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
pairs = [(a, b) for a in range(-L, L + 1) for b in range(-L, L + 1) if (a, b) > (0, 0)]
rng = np.random.default_rng(0)

def some_features(X, chosen):
    cols = [np.ones(len(X))]
    for a, b in chosen:
        phase = a * X[:, 0] + b * X[:, 1]
        cols += [np.cos(phase), np.sin(phase)]
    return np.stack(cols, axis=1)

for K in (2, 4, 8, 16, 24):
    accs = []
    for repeat in range(5):
        chosen = [pairs[i] for i in rng.choice(len(pairs), size=K, replace=False)]
        model = GridSearchCV(LogisticRegression(max_iter=5000), {"C": [0.1, 1, 10, 100]}, cv=5)
        model.fit(some_features(X_train, chosen), y_train)
        accs.append(model.score(some_features(X_test, chosen), y_test))
    print(f"K = {K:2d} random pairs ({2 * K + 1:2d} features): test accuracy {np.mean(accs):.3f} ± {np.std(accs):.3f}")